# 5.3 混合并行策略 (Hybrid Parallelism)

实际大规模训练中，单一并行方式无法满足需求，需要组合多种并行策略。

本节涵盖：
- 3D并行
- 自动并行

## 1. 3D并行

**目的**：结合DP+TP+PP训练超大模型

**基本原理**：节点内使用TP（利用NVLink高带宽），节点间使用PP（利用InfiniBand），全局使用DP，三者组合实现万卡级训练。

**3D并行布局**：
- **TP组**：同一节点内的GPU，通过NVLink高速互联
- **PP组**：跨节点的GPU，通过InfiniBand互联
- **DP组**：拥有相同模型阶段的GPU组，梯度同步

**代表**：Megatron-Deepspeed，用于训练GPT-3等超大模型

**通信模式**：
- TP：AllReduce（节点内，高带宽）
- PP：点对点（节点间，中等带宽）
- DP：AllReduce（跨节点，较低带宽）

In [ ]:
import torch

total_gpus = 64
tp_size = 8
pp_size = 4
dp_size = total_gpus // (tp_size * pp_size)

print('=== 3D Parallelism ===')
print(f'Total GPUs: {total_gpus}')
print(f'Tensor Parallel (TP): {tp_size} (within node, NVLink)')
print(f'Pipeline Parallel (PP): {pp_size} (across nodes, InfiniBand)')
print(f'Data Parallel (DP): {dp_size} (gradient sync)')
print(f'Verify: {tp_size} x {pp_size} x {dp_size} = {tp_size*pp_size*dp_size}')

model_params_b = 175
bytes_per_param = 2
model_size_gb = model_params_b * 1e9 * bytes_per_param / 1e9

print(f'\nExample: {model_params_b}B parameter model (BF16)')
print(f'  Total model size: {model_size_gb:.0f} GB')
print(f'  Per TP group (split by {tp_size}): {model_size_gb/tp_size:.0f} GB')
print(f'  Per PP stage (split by {pp_size}): {model_size_gb/pp_size:.0f} GB')
print(f'  Per GPU (TP+PP): {model_size_gb/tp_size/pp_size:.0f} GB')

print(f'\nGPU layout (example with 64 GPUs):')
print(f'  Node 0: GPUs 0-7  -> TP group 0, PP stage 0')
print(f'  Node 1: GPUs 8-15 -> TP group 0, PP stage 1')
print(f'  Node 2: GPUs 16-23 -> TP group 0, PP stage 2')
print(f'  Node 3: GPUs 24-31 -> TP group 0, PP stage 3')
print(f'  Nodes 4-7: DP replica of nodes 0-3')
print(f'\nKey: 3D parallelism is the standard for training 100B+ models.')

## 2. 自动并行

**目的**：自动搜索最优并行策略

**基本原理**：编译器根据计算图和集群拓扑自动分析最优的并行切分方案，减少人工调参成本。

**自动并行的核心问题**：
- 给定模型和集群配置，找到最优的并行策略
- 优化目标：最小化训练时间（计算+通信）
- 约束：每张GPU的显存不超过上限

**代表框架**：
- **Alpa**：自动并行编译器，自动搜索最优并行策略
- **Galvatron**：自动选择DP/TP/PP的最优组合

**搜索空间**：
- 每层可以选择不同的并行策略
- 不同层的切分方式可以不同
- 需要考虑层间通信开销

In [ ]:
import torch

print('=== Auto Parallelism ===')
print(f'\nManual parallelism requires experts to decide:')
print(f'  - TP size, PP size, DP size')
print(f'  - Which layers to parallelize and how')
print(f'  - Communication-computation overlap strategy')
print(f'\nAuto parallelism automates this decision:')

strategies = {
    'Attention layer (large)': 'TP across NVLink (compute-bound)',
    'FFN layer (large)': 'TP across NVLink (compute-bound)',
    'Embedding layer': 'DP + TP (memory-bound)',
    'LayerNorm': 'SP (element-wise, memory savings)',
    'Loss computation': 'DP (simple, no sharding needed)',
}

print(f'\nExample auto-parallel decisions for a Transformer layer:')
for layer, strategy in strategies.items():
    print(f'  {layer}: {strategy}')

print(f'\nOptimization objectives:')
print(f'  1. Minimize total training time (compute + communication)')
print(f'  2. Ensure each GPU memory usage < limit')
print(f'  3. Balance computation across GPUs')
print(f'\nKey: Auto parallelism reduces the need for manual tuning,')
print(f'making distributed training accessible to more practitioners.')